# QuantumXCT — GRN / LR search benchmark

Benchmark the topology search across the three biological scopes and report, for each, the KL-reduction summary plus two contribution tables.

## Scopes

* **GRN + LR** (`scope='all'`) — search every candidate link (intra- and inter-cellular), density-matrix screened.
* **GRN only** (`scope='intra'`) — within-cell links only, density-matrix screened, optimized **one cell at a time** (cell 1 to convergence, then cell 2). LR (cross-cell) links may not contribute.
* **LR only** (`scope='inter'`) — cross-cell (ligand-receptor) links **only**: *every* cross-register pair, both directions, with **no density-matrix screening** and **no GRN (intra-cell) links** at all.

## Contribution tables (per scope, for the final optimized topology)

* **Cumulative ablation reduction** — add gates back in order; running KL and cumulative reduction from baseline (order-dependent).
* **Isolated / leave-one-out contribution** — each gate's isolated effect (gate alone) and marginal effect (full circuit minus that gate); order-independent.

Uses synthetic data by default — replace `load_counts()` with your own joint histograms (e.g. from `qxct.encoding.create_joint_histogram` on binarized scRNA-seq matrices).

In [ ]:
import os
import numpy as np
import pandas as pd

from qxct import RunConfig, EncodingConfig, BackendConfig, SearchConfig, OptimizerConfig
from qxct.analysis import compare_scopes

OUT_DIR = "results/benchmark_grn_lr"
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Data

3 genes in cell 1 (q0, q1, q2) and 2 genes in cell 2 (q3, q4).

In [ ]:
def load_counts():
    ct1_basal = {"000": 60, "001": 15, "010": 10, "011": 5, "100": 5, "101": 3, "110": 2, "111": 0}
    ct2_basal = {"00": 70, "01": 10, "10": 15, "11": 5}
    ct1_inter = {"000": 15, "001": 25, "010": 20, "011": 15, "100": 10, "101": 8, "110": 5, "111": 2}
    ct2_inter = {"00": 15, "01": 35, "10": 20, "11": 30}
    gene_list = ["A_lig", "A_tf", "A_rec", "B_rec", "B_tf"]   # q0..q2 = cell1 ; q3,q4 = cell2
    return ct1_basal, ct2_basal, ct1_inter, ct2_inter, gene_list

ct1_b, ct2_b, ct1_i, ct2_i, genes = load_counts()

## 2. Configuration

In [ ]:
config = RunConfig(
    encoding=EncodingConfig(norm="l1"),
    backend=BackendConfig(shots=3000),
    search=SearchConfig(
        algorithm="multi_epoch", threshold=0.02, search_mode="abs",
        n_epochs=None, kl_tol=0.02, max_greedy_depth=8,
    ),
    optimizer=OptimizerConfig(method="COBYLA", initial_angle=np.pi / 2),
)

## 3. Run all three scopes

`compare_scopes` runs the full search + angle optimization for each scope and returns a summary DataFrame plus per-scope detail (topology, angles, and both tables).

In [ ]:
bench = compare_scopes(config, ct1_b, ct2_b, ct1_i, ct2_i, gene_list=genes)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

## 4. Scope summary — KL reduction by search scope

In [ ]:
bench["summary"].to_csv(os.path.join(OUT_DIR, "scope_summary.csv"), index=False)
bench["summary"]

## 5. Per-scope tables

Cumulative ablation reduction and isolated / leave-one-out contribution for each scope. CSVs are written to `./benchmark_out/`.

In [ ]:
# --- self-contained: (re)compute `bench` if it is not in the kernel, so this
# cell works on its own even after a kernel restart or out-of-order run ---
try:
    bench
except NameError:
    bench = compare_scopes(config, ct1_b, ct2_b, ct1_i, ct2_i, gene_list=genes)

for scope, res in bench["detail"].items():
    tag = {"all": "GRN+LR", "intra": "GRN-only", "inter": "LR-only"}[scope]
    print("=" * 72)
    print(f"[{tag}]  {res['n_gates']} gates | baseline KL {res['baseline_kl']:.4f} "
          f"-> final KL {res['final_kl']:.4f}  (reduction {res['reduction']:.4f}, "
          f"{res['pct_reduction']:.1f}%)")
    print("=" * 72)
    print("\nCumulative ablation reduction:")
    print(res["cumulative_ablation"].to_string(index=False))
    print("\nIsolated / leave-one-out gate contribution:")
    print(res["isolated_contribution"].to_string(index=False))
    print()
    res["cumulative_ablation"].to_csv(os.path.join(OUT_DIR, f"{scope}_cumulative_ablation.csv"), index=False)
    res["isolated_contribution"].to_csv(os.path.join(OUT_DIR, f"{scope}_isolated_contribution.csv"), index=False)

### Inspect a single scope's tables interactively

In [ ]:
bench["detail"]["all"]["cumulative_ablation"]

In [ ]:
bench["detail"]["inter"]["isolated_contribution"]